In [ ]:
# %matplotlib ipympl

In [ ]:
from pathlib import Path

import neurokit2 as nk
import numpy as np
import pandas as pd
import scipy.io
import scipy.signal
from matplotlib import pyplot as plt
from tqdm import tqdm

from config import Config as c

In [ ]:
df_subjects = pd.read_csv(c.SUBJECTS_PATH)

In [ ]:
df_subjects['patient_file'] = df_subjects.apply(lambda r: c.PPG_DIR / f'p{r["subject_id"]:06d}_{r["segment_id"]}.mat', axis=1)

In [ ]:
subject_id, category, patient_file = df_subjects.loc[0, ['subject_id', 'class', 'patient_file']]

In [ ]:
fs = 125

In [ ]:
mat = scipy.io.loadmat(patient_file)

In [ ]:
t = mat['time'].squeeze()
ppg_raw = mat['data'].squeeze()

In [ ]:
ppg_none = nk.ppg.ppg_clean(ppg_signal=ppg_raw, sampling_rate=fs, method="none")

In [ ]:
fig, axes = plt.subplots(figsize=(14, 8))
plt.plot(t, ppg_raw, t, ppg_none)
plt.legend(['ppg_raw', 'ppg_none'])
plt.xlim(0, 10)

In [ ]:
ppg_elgendi = nk.ppg.ppg_clean(ppg_signal=ppg_raw, sampling_rate=fs)

In [ ]:
fig, axes = plt.subplots(figsize=(14, 8))
plt.plot(t, ppg_raw, t, ppg_elgendi)
plt.legend(['ppg_raw', 'ppg_elgendi'])
plt.xlim(0, 10)

In [ ]:
b, a = scipy.signal.cheby2(4, 20, 30, fs=fs)
ppg_filt = scipy.signal.filtfilt(b, a, ppg_raw)

In [ ]:
fig, axes = plt.subplots(figsize=(14, 8))
plt.plot(t, ppg_raw, t, ppg_filt)
plt.legend(['ppg_raw', 'ppg_filt'])
plt.xlim(0, 10)

In [ ]:
ppg_points = nk.ppg.ppg_peaks(ppg_cleaned=ppg_filt, sampling_rate=fs)
ppg_points

In [ ]:
fig, axes = plt.subplots(figsize=(14, 8))
plt.plot(t, ppg_filt, color='C0')
plt.scatter(t[ppg_points[1]['PPG_Peaks']], ppg_filt[ppg_points[1]['PPG_Peaks']], color='C1')
# plt.legend(['ppg_filt'])
plt.xlim(0, 60)

In [ ]:
ppg_points = nk.ppg.ppg_peaks(ppg_cleaned=ppg_filt, sampling_rate=fs, method="charlton")
ppg_points

In [ ]:
fig, axes = plt.subplots(figsize=(14, 8))
plt.plot(t, ppg_filt, color='C0')
plt.scatter(t[ppg_points[1]['PPG_Peaks']], ppg_filt[ppg_points[1]['PPG_Peaks']], color='C1')
plt.scatter(t[ppg_points[1]['PPG_Onsets']], ppg_filt[ppg_points[1]['PPG_Onsets']], color='C2')
# plt.scatter(t[ppg_points[1]['PPG_Peaks_Unfixed']], ppg_filt[ppg_points[1]['PPG_Peaks_Unfixed']], color='C3')
# plt.scatter(t[ppg_points[1]['PPG_Onsets_Unfixed']], ppg_filt[ppg_points[1]['PPG_Onsets_Unfixed']], color='C4')
# plt.legend(['ppg_filt'])
plt.xlim(0, 10)
plt.grid()

In [ ]:
vpg = np.gradient(ppg_filt, t)

In [ ]:
ppg_points2 = nk.ppg.ppg_peaks(ppg_cleaned=vpg, sampling_rate=fs, method="charlton")
ppg_points2

In [ ]:
fig, axes = plt.subplots(figsize=(14, 8))
plt.plot(t, vpg, color='C0')
plt.scatter(t[ppg_points2[1]['PPG_Peaks']], vpg[ppg_points2[1]['PPG_Peaks']], color='C1')
plt.scatter(t[ppg_points2[1]['PPG_Onsets']], vpg[ppg_points2[1]['PPG_Onsets']], color='C2')
# plt.scatter(t[ppg_points[1]['PPG_Peaks_Unfixed']], ppg_filt[ppg_points[1]['PPG_Peaks_Unfixed']], color='C3')
# plt.scatter(t[ppg_points[1]['PPG_Onsets_Unfixed']], ppg_filt[ppg_points[1]['PPG_Onsets_Unfixed']], color='C4')
# plt.legend(['ppg_filt'])
plt.xlim(0, 10)
plt.grid()